# QR Code Generator for Vocabulary App\n\nThis notebook generates QR codes for each URL parameter such as `?uid=u_001`.\nUpload the generated QR codes wherever you want to distribute the vocabulary app.

In [ ]:
# QR Code Generator for English-German Vocabulary Book\n# Run this code in Google Colab.\n\n# 1) Install QR library\n!pip install -q qrcode[pil]\n\n# 2) Import libraries\nimport os\nimport zipfile\nimport secrets\nfrom pathlib import Path\n\nimport pandas as pd\nimport qrcode\nfrom google.colab import files\n\n# 3) Set your GitHub Pages URL\n# Change this URL to your own GitHub Pages URL.\nBASE_URL = "https://bokuhabobu.github.io/present/"\n\n# 4-A) Manual user IDs\nUSER_IDS = [\n    "u_001",\n    "u_002",\n    "u_003",\n    "u_004",\n    "u_005",\n]\n\n# 4-B) Optional: random user IDs\n# If you want random IDs, uncomment the next line and comment out the manual USER_IDS above.\n# USER_IDS = [f"u_{secrets.token_hex(3)}" for _ in range(20)]\n\n# 5) Create output folder\noutput_dir = Path("qr_codes")\noutput_dir.mkdir(exist_ok=True)\n\nrecords = []\n\n# 6) Generate QR images\nfor uid in USER_IDS:\n    url = f"{BASE_URL}?uid={uid}"\n\n    qr = qrcode.QRCode(\n        version=None,\n        error_correction=qrcode.constants.ERROR_CORRECT_M,\n        box_size=10,\n        border=4,\n    )\n\n    qr.add_data(url)\n    qr.make(fit=True)\n\n    img = qr.make_image(fill_color="black", back_color="white")\n\n    filename = f"{uid}.png"\n    filepath = output_dir / filename\n    img.save(filepath)\n\n    records.append({\n        "uid": uid,\n        "url": url,\n        "qr_file": filename,\n    })\n\n# 7) Save uid-url mapping table\nmapping_df = pd.DataFrame(records)\nmapping_df.to_csv("qr_mapping.csv", index=False, encoding="utf-8-sig")\n\nprint(mapping_df)\n\n# 8) Zip all QR images and mapping CSV\nzip_filename = "qr_codes.zip"\n\nwith zipfile.ZipFile(zip_filename, "w", compression=zipfile.ZIP_DEFLATED) as zipf:\n    for png_file in output_dir.glob("*.png"):\n        zipf.write(png_file, arcname=png_file.name)\n\n    zipf.write("qr_mapping.csv", arcname="qr_mapping.csv")\n\n# 9) Download zip file\nfiles.download(zip_filename)\n